# Part 3 and 4 - Exact Tokenization and Tokenized Session Sequences

## Exact Tokenization and the Decision Not to Semantically Canonicalize

The source owner defines every distinct `segment_name` and `screen_name` string as a genuinely
different component or segment. Semantic canonicalization would therefore destroy source-defined
information. Values such as `Home`, `HomeVC`, `android/Home`, suffixes, controller names, paths,
URLs, punctuation, and casing are preserved exactly.

Each event token is the exact tuple `(event_type, segment_name, screen_name)`. Duration and
identifiers are not part of the categorical token.


## Configuration and imports


In [3]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

WORKING_DIR = Path.cwd()
REPO_ROOT = WORKING_DIR if (WORKING_DIR / "data" / "final_clean_events.csv").exists() else WORKING_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.exact_event_analysis import (
    ACTION_EVENT_TYPES,
    INACTIVITY_THRESHOLDS_MINUTES,
    MISSING_SENTINEL,
    REQUIRED_COLUMNS,
    SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    VIEW_EVENT_TYPES,
    add_duration_fields,
    add_exact_tokens,
    add_time_fields,
    add_token_ids,
    availability_by_event_type,
    build_session_sequences,
    build_token_dictionary,
    component_by_event_type,
    construct_clean_sessions,
    exact_tuple_counts,
    file_fingerprint,
    inferred_dtypes,
    input_csv_path,
    markdown_table,
    masked_sample,
    missing_summary,
    original_order_timestamp_issues,
    original_session_summary,
    quantile_table,
    read_events,
    representative_sequences,
    safe_json,
    save_token_outputs,
    sort_events,
    threshold_comparison,
    validate_required_columns,
    validate_tokenization,
    value_counts_with_pct,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
np.random.seed(42)

INPUT_CSV = input_csv_path(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Input CSV: {INPUT_CSV.relative_to(REPO_ROOT)}")


Repository root: D:\BehaviourClassification
Input CSV: data\final_clean_events.csv


## Load, validate, parse, sort, and derive clean sessions

Clean sessions are derived separately as `clean_session_id`. The original `session_id` remains
unchanged and traceable.


In [5]:
before_fingerprint = file_fingerprint(INPUT_CSV)
source_events = read_events(INPUT_CSV)
print(f"Rows after source load: {len(source_events):,}")
validate_required_columns(source_events)
events, time_metadata = add_time_fields(source_events)
print(f"Rows after timestamp parsing: {len(events):,}")
events = add_duration_fields(events)
print(f"Rows after duration derivation: {len(events):,}")
ordered_events = sort_events(events)
print(f"Rows after deterministic ordering: {len(ordered_events):,}")
ordered_events = construct_clean_sessions(ordered_events, SELECTED_INACTIVITY_THRESHOLD_MINUTES)
print(f"Rows after clean-session derivation: {len(ordered_events):,}")
print(safe_json({
    "selected_timestamp_unit": time_metadata["selected_timestamp_unit"],
    "primary_event_time_column": time_metadata["primary_event_time_column"],
    "selected_inactivity_threshold_minutes": SELECTED_INACTIVITY_THRESHOLD_MINUTES,
}))


Rows after source load: 114,534
Rows after timestamp parsing: 114,534
Rows after duration derivation: 114,534
Rows after deterministic ordering: 114,534
Rows after clean-session derivation: 114,534
{
  "selected_timestamp_unit": "ms",
  "primary_event_time_column": "timestamp_datetime",
  "selected_inactivity_threshold_minutes": 30
}


## Token serialization

Tokens are serialized as JSON arrays to avoid unsafe delimiter concatenation. For derived token
serialization only, missing fields are represented by `<MISSING>`. Original source columns remain
missing and are not filled.


In [7]:
tokenized_base = add_exact_tokens(ordered_events, MISSING_SENTINEL)
print(f"Rows after exact-token serialization: {len(tokenized_base):,}")
print("Example serialized tokens:")
print(markdown_table(tokenized_base[["event_type", "segment_name", "screen_name", "exact_token"]].head(10), max_rows=10))


Rows after exact-token serialization: 114,534
Example serialized tokens:
| event_type | segment_name | screen_name | exact_token |
| --- | --- | --- | --- |
| View | SplashVC | <MISSING> | ["View","SplashVC","<MISSING>"] |
| View | SplashVC | <MISSING> | ["View","SplashVC","<MISSING>"] |
| View | MainTabBarController | <MISSING> | ["View","MainTabBarController","<MISSING>"] |
| View | HomeGuestVC | <MISSING> | ["View","HomeGuestVC","<MISSING>"] |
| Action | guest/Home/Nav_profile | HomeGuestVC | ["Action","guest/Home/Nav_profile","HomeGuestVC"] |
| View | AccountHomeController | <MISSING> | ["View","AccountHomeController","<MISSING>"] |
| View | HomeGuestVC | <MISSING> | ["View","HomeGuestVC","<MISSING>"] |
| Action | guest/Account-Login-Guest/Buttons/Click_login | AccountHomeController | ["Action","guest/Account-Login-Guest/Buttons/Click_login","AccountHomeController"] |
| View | BaseNavigation | <MISSING> | ["View","BaseNavigation","<MISSING>"] |
| View | LoginVC | <MISSING> | ["View

## Deterministic token dictionary

Token IDs are assigned by sorting the exact JSON token strings and numbering them from 1. Python's
runtime hash is not used.


In [9]:
token_dictionary = build_token_dictionary(tokenized_base)
tokenized_events = add_token_ids(tokenized_base, token_dictionary)
print(f"Rows after token ID assignment: {len(tokenized_events):,}")
print(f"Token vocabulary size: {len(token_dictionary):,}")
print("\nToken dictionary preview:")
print(markdown_table(token_dictionary.head(20), max_rows=20))
print("\nMost frequent tokens:")
print(markdown_table(token_dictionary.sort_values("total_event_frequency", ascending=False).head(20), max_rows=20))


Rows after token ID assignment: 114,534
Token vocabulary size: 3,338

Token dictionary preview:
| token_id | exact_token | event_type | segment_name | screen_name | total_event_frequency | original_session_frequency | clean_session_frequency | first_observed_timestamp | last_observed_timestamp |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | ["Action","/","DeviceDetailVC"] | Action | / | DeviceDetailVC | 1 | 1 | 1 | 2026-07-07 04:38:37.692000+00:00 | 2026-07-07 04:38:37.692000+00:00 |
| 2 | ["Action","/","HomeVC"] | Action | / | HomeVC | 1 | 1 | 1 | 2026-07-02 10:19:07.540000+00:00 | 2026-07-02 10:19:07.540000+00:00 |
| 3 | ["Action","/","ModemRestartScheduleVC"] | Action | / | ModemRestartScheduleVC | 9 | 5 | 5 | 2026-07-01 08:00:32.971000+00:00 | 2026-07-06 04:48:37.295000+00:00 |
| 4 | ["Action","/","ServiceManageVC"] | Action | / | ServiceManageVC | 2 | 2 | 2 | 2026-07-02 10:29:34.188000+00:00 | 2026-07-06 04:42:45.638000+00:00 |
| 5 | ["Action","/Body/BackButt

## Tokenized event table

The tokenized event table retains every source column and adds derived context fields, including
previous/next token IDs, exact-repeat flags, previous-view context, gap fields, duration-derived
fields, and missingness flags. Previous-view context does not alter the exact source token.


In [11]:
tokenized_preview_columns = [
    "record_id", "session_id", "clean_session_id", "event_index_in_original_session",
    "event_index_in_clean_session", "event_type", "segment_name", "screen_name",
    "token_id", "exact_token", "previous_token_id", "next_token_id",
    "time_gap_from_previous_seconds", "time_gap_to_next_seconds",
    "is_consecutive_exact_repeat", "previous_view_token_id", "seconds_since_previous_view",
    "duration_numeric", "log1p_duration", "duration_missing", "duration_zero",
    "duration_outlier", "segment_name_missing", "screen_name_missing",
]
print(markdown_table(tokenized_events[tokenized_preview_columns].head(20), max_rows=20))


| record_id | session_id | clean_session_id | event_index_in_original_session | event_index_in_clean_session | event_type | segment_name | screen_name | token_id | exact_token | previous_token_id | next_token_id | time_gap_from_previous_seconds | time_gap_to_next_seconds | is_consecutive_exact_repeat | previous_view_token_id | seconds_since_previous_view | duration_numeric | log1p_duration | duration_missing | duration_zero | duration_outlier | segment_name_missing | screen_name_missing |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 6a471aa6a831463d3a03962f | 000A5106-BBC0-45BA-AC04-53BAF35B99F3 | 000A5106-BBC0-45BA-AC04-53BAF35B99F3::clean_0 | 0 | 0 | View | SplashVC | <MISSING> | 2668 | ["View","SplashVC","<MISSING>"] | <MISSING> | 2668.0 | <MISSING> | 1.122 | False | <MISSING> | <MISSING> | 0 | 0.0 | False | True | False | False | True |
| 6a471aa52eed2dfdb6c9c2d0 | 000A5106-BBC0-4

## Tokenized session sequences

Ordered raw token sequences and run-length compressed sequences are created for both original
sessions and derived clean sessions.


In [13]:
original_sequences = build_session_sequences(tokenized_events, "session_id")
clean_sequences = build_session_sequences(tokenized_events, "clean_session_id")
print(f"Original-session sequences: {len(original_sequences):,}")
print(f"Clean-session sequences: {len(clean_sequences):,}")
print("\nOriginal sequence metrics:")
print(markdown_table(original_sequences[[
    "session_id", "raw_event_count", "compressed_event_count", "unique_exact_token_count",
    "view_count", "action_count", "missing_segment_count", "missing_screen_count",
    "session_duration_seconds", "maximum_inter_event_gap_seconds",
    "exact_repeat_count", "compression_ratio_compressed_to_raw",
]].head(20), max_rows=20))
print("\nCompression statistics:")
print(markdown_table(quantile_table(original_sequences["compression_ratio_compressed_to_raw"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))


Original-session sequences: 1,425
Clean-session sequences: 1,739

Original sequence metrics:
| session_id | raw_event_count | compressed_event_count | unique_exact_token_count | view_count | action_count | missing_segment_count | missing_screen_count | session_duration_seconds | maximum_inter_event_gap_seconds | exact_repeat_count | compression_ratio_compressed_to_raw |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 000A5106-BBC0-45BA-AC04-53BAF35B99F3 | 31 | 29 | 22 | 24 | 7 | 0 | 24 | 66.592 | 26.554 | 2 | 0.935484 |
| 0096BDF5-AFCF-4367-B5AB-6FA86F6F3CD3 | 10 | 9 | 3 | 10 | 0 | 0 | 10 | 54.711 | 50.801 | 1 | 0.9 |
| 00c93be9-5085-49d0-b84e-1a1ce50ec112 | 117 | 100 | 29 | 73 | 44 | 0 | 73 | 313.228 | 61.711 | 17 | 0.854701 |
| 0101d1a0-8c7a-478c-9f6b-858df3f0ea40 | 230 | 159 | 65 | 113 | 117 | 0 | 113 | 3069.062 | 710.75 | 71 | 0.691304 |
| 01923B7E-C036-44BF-BC09-D17BE54C781A | 29 | 26 | 14 | 24 | 5 | 0 | 24 | 184.158 | 123.093 | 3 | 0.896552 |
| 019a8b0

## Representative readable sequences

Representative examples are displayed with token IDs and readable exact JSON tokens. Long previews
are truncated only for display.


In [15]:
original_examples = representative_sequences(original_sequences, tokenized_events, "session_id", token_dictionary)
clean_examples = representative_sequences(clean_sequences, tokenized_events, "clean_session_id", token_dictionary)
print("Original-session examples:")
print(markdown_table(original_examples, max_rows=10))
print("\nClean-session examples:")
print(markdown_table(clean_examples, max_rows=10))


Original-session examples:
| example_type | session_id | raw_event_count | compressed_event_count | compression_ratio_compressed_to_raw | raw_token_sequence_preview | readable_sequence_preview |
| --- | --- | --- | --- | --- | --- | --- |
| short_session | 67be4e4e-bc26-4e9b-b84b-ccb3611d1132 | 1 | 1 | 1.0 | [3330] | ['["View","loylaty_promotion_view","<MISSING>"]'] |
| median_length_session | 3F409BEE-1A91-4FC6-ABEE-676C31F76FB6 | 34 | 32 | 0.941176 | [2668, 2668, 2606, 2587, 2642, 2642, 1252, 259, 2664, 2606, 2587, 1147, 2607, 2664, 1153, 2613, 2607, 1156, 2610, 2613] | ['["View","SplashVC","<MISSING>"]', '["View","SplashVC","<MISSING>"]', '["View","MainTabBarController","<MISSING>"]', '["View","HomeVC","<MISSING>"]', '["View","PopupBigMessageVC","<MISSING>"]', '["View","PopupBigMessageVC","<MISSING>"]', '["Action","invite_update_close","HomeVC"]', '["Action","Home/other_manage_internet","HomeVC"]', '["View","ServiceManageVC","<MISSING>"]', '["View","MainTabBarController","<MISSING>"

## Validation

These assertions confirm exact-value preservation, one-to-one token mapping, reversible
serialization, deterministic ordering, count reconciliation, sequence validity, and repeated-run
token dictionary stability.


In [17]:
validation_results = validate_tokenization(
    source_events,
    tokenized_events,
    token_dictionary,
    original_sequences,
    clean_sequences,
)
after_fingerprint = file_fingerprint(INPUT_CSV)
assert before_fingerprint == after_fingerprint, "Input CSV was modified."
validation_results["input_file_status"] = "unchanged"
print("Validation results:")
print(safe_json(validation_results))


Validation results:
{
  "source_rows": 114534,
  "tokenized_rows": 114534,
  "token_vocabulary_size": 3338,
  "original_sequence_count": 1425,
  "clean_sequence_count": 1739,
  "validation_status": "passed",
  "input_file_status": "unchanged"
}


## Save tokenized outputs

After validation passes, save the deterministic token dictionary and tokenized session sequences
into repo-relative output files. Sequence files are JSON Lines so raw token arrays and compressed
repeat objects are preserved without delimiter ambiguity.


In [19]:
OUTPUT_DIR = REPO_ROOT / "outputs"
output_manifest = save_token_outputs(
    token_dictionary=token_dictionary,
    original_sequences=original_sequences,
    clean_sequences=clean_sequences,
    output_dir=OUTPUT_DIR,
)
print("Saved tokenized outputs:")
print(markdown_table(pd.DataFrame(output_manifest["files"]), max_rows=10))


Saved tokenized outputs:
| name | path | rows |
| --- | --- | --- |
| token_dictionary.csv | D:\BehaviourClassification\outputs\token_dictionary.csv | 3338 |
| token_dictionary.jsonl | D:\BehaviourClassification\outputs\token_dictionary.jsonl | 3338 |
| original_session_sequences.jsonl | D:\BehaviourClassification\outputs\original_session_sequences.jsonl | 1425 |
| clean_session_sequences.jsonl | D:\BehaviourClassification\outputs\clean_session_sequences.jsonl | 1739 |
| tokenized_output_manifest.json | D:\BehaviourClassification\outputs\tokenized_output_manifest.json | 1 |


## Final summary

This notebook stops after exact tokenization, duplicate-aware sequence construction, and validation.
It does not implement journey-candidate discovery, PrefixSpan, embeddings, clustering, journey
labeling, or cluster evaluation.


In [21]:
duplicate_stats = {
    "duplicate_record_id_rows": int(source_events["record_id"].duplicated(keep=False).sum()),
    "duplicate_record_id_excess": int(source_events["record_id"].duplicated().sum()),
    "fully_duplicated_rows": int(source_events.drop(columns=["source_row_number"]).duplicated().sum()),
}
compression_stats = {
    "mean_compression_ratio_compressed_to_raw": float(original_sequences["compression_ratio_compressed_to_raw"].mean()),
    "median_compression_ratio_compressed_to_raw": float(original_sequences["compression_ratio_compressed_to_raw"].median()),
    "sessions_with_repeated_exact_tokens": int(original_sequences["exact_repeat_count"].gt(0).sum()),
}
final_summary = {
    "dataset_size": int(len(source_events)),
    "token_vocabulary_size": int(len(token_dictionary)),
    "original_sessions": int(original_sequences.shape[0]),
    "clean_sessions": int(clean_sequences.shape[0]),
    "selected_inactivity_threshold_minutes": SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    "duplicate_event_statistics": duplicate_stats,
    "compression_statistics": compression_stats,
    "validation_status": validation_results["validation_status"],
    "saved_output_files": [file_info["name"] for file_info in output_manifest["files"]],
    "important_unresolved_questions": [
        "Whether the 30-minute inactivity threshold should be changed by business policy.",
        "Whether timestamp and created_at disagreement has source-system meaning.",
        "Whether duration units are formally documented by the source owner.",
    ],
}
print(safe_json(final_summary))


{
  "dataset_size": 114534,
  "token_vocabulary_size": 3338,
  "original_sessions": 1425,
  "clean_sessions": 1739,
  "selected_inactivity_threshold_minutes": 30,
  "duplicate_event_statistics": {
    "duplicate_record_id_rows": 0,
    "duplicate_record_id_excess": 0,
    "fully_duplicated_rows": 0
  },
  "compression_statistics": {
    "mean_compression_ratio_compressed_to_raw": 0.8336797136842106,
    "median_compression_ratio_compressed_to_raw": 0.864865,
    "sessions_with_repeated_exact_tokens": 1350
  },
  "validation_status": "passed",
  "saved_output_files": [
    "token_dictionary.csv",
    "token_dictionary.jsonl",
    "original_session_sequences.jsonl",
    "clean_session_sequences.jsonl",
    "tokenized_output_manifest.json"
  ],
  "important_unresolved_questions": [
    "Whether the 30-minute inactivity threshold should be changed by business policy.",
    "Whether timestamp and created_at disagreement has source-system meaning.",
    "Whether duration units are formally d